# Sheet 7 — Real music on the Tonnetz (Chapter 8)

**Data.** Run `bash scripts/fetch_data.sh` once from the repository root. It downloads
* the Annotated Beethoven Corpus (DCMLab/ABC, CC BY-NC-SA 4.0),
* NYU MARL chord labels for RWC-Pop (100 songs) and a USPop2002 subset (195 songs),

and uses the Bach chorales that ship with `music21`. Data are *not* redistributed in this repository.

In [1]:
# Setup: make the repository importable whether or not `pip install -e .` was run
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np, networkx as nx, matplotlib.pyplot as plt
from tonnetzkit import *
%matplotlib inline
import json, collections, pandas as pd

In [2]:
DATA = os.path.abspath('../data/external')
print('available:', os.listdir(DATA) if os.path.isdir(DATA) else 'run scripts/fetch_data.sh first')

available: ['Chord-Annotations', 'bach_triads.json', 'ABC']


### Parsing chord syntaxes

In [3]:
for lab in ['C:maj', 'A:min7', 'G:7/3', 'Bb:maj9', 'F#:hdim7', 'D:(1,b3,5)', 'N']:
    print(f'{lab:12s} pcs={sorted(harte_to_pcs(lab)) if harte_to_pcs(lab) else None}  triad={harte_to_triad(lab)}')

C:maj        pcs=[0, 4, 7]  triad=C
A:min7       pcs=[0, 4, 7, 9]  triad=Am
G:7/3        pcs=[2, 5, 7, 11]  triad=G
Bb:maj9      pcs=[0, 2, 5, 9, 10]  triad=A#
F#:hdim7     pcs=[0, 4, 6, 9]  triad=None
D:(1,b3,5)   pcs=[2, 5, 9]  triad=D
N            pcs=None  triad=None


### Run the full study (same code as `scripts/corpus_analysis.py`)

In [4]:
import importlib.util
spec = importlib.util.spec_from_file_location('ca', '../scripts/corpus_analysis.py'); ca = importlib.util.module_from_spec(spec); spec.loader.exec_module(ca)
G, D = plr_graph(), plr_distance_table()
corpora = {
  'Bach chorales': list(ca.bach_sequences(cache=os.path.join(DATA, 'bach_triads.json')).values()),
  'Beethoven (ABC)': [ca.dedupe([dcml_to_triad(t) for t in v]) for v in load_dcml_corpus(os.path.join(DATA, 'ABC', 'harmonies')).values()],
  'RWC Pop': [sequence_to_triads(v) for v in load_lab_corpus(os.path.join(DATA, 'Chord-Annotations', 'RWC_Pop_Chords')).values()],
  'USPop': [sequence_to_triads(v) for v in load_lab_corpus(os.path.join(DATA, 'Chord-Annotations', 'uspopLabels')).values()],
}
res = [ca.analyse(k, v, D, G, n_shuffles=10) for k, v in corpora.items()]
pd.DataFrame([{k: r[k] for k in ('corpus', 'pieces', 'transitions', 'mean_plr', 'null_mean_plr')} | {'P(d=1)': r['dist'].get(1, 0), 'P(d=4)': r['dist'].get(4, 0)} for r in res]).round(3)

,corpus,pieces,transitions,mean_plr,null_mean_plr,P(d=1),P(d=4)
0,Bach chorales,400,13794,2.329,2.366,0.170,0.072
1,Beethoven (ABC),70,15097,2.394,2.577,0.131,0.075
2,RWC Pop,100,11737,2.618,2.467,0.126,0.215
3,USPop,194,21945,2.486,2.442,0.154,0.197


### What are the distance-4 moves in pop?

In [5]:
cnt = collections.Counter(); n = 0
for s in corpora['USPop']:
    for a, b in zip(s, s[1:]):
        if D[a][b] == 4: cnt[((b.root - a.root) % 12, a.quality + b.quality)] += 1; n += 1
[(k, round(v/n, 3)) for k, v in cnt.most_common(5)]

[((2, 'MM'), 0.45),
 ((10, 'MM'), 0.315),
 ((11, 'MM'), 0.078),
 ((10, 'mm'), 0.065),
 ((2, 'mm'), 0.046)]

Root motion by a whole step between two major triads (e.g. IV→V, ♭VII→I) accounts for most of them. In the PLR metric
F→G costs 4 steps, although it is idiomatic in rock (Biamonte 2010; de Clercq & Temperley 2011). The metric is a *model*, not a verdict.

## Exercises
1. Replace the PLR distance with `voice_leading_distance`. Do the corpus rankings change?
2. Restrict the pop corpora to songs whose chords are ≥ 90 % triad-reducible. Does the result persist?
3. Compute, per piece, the fraction of the 24 triads visited. Which corpus is most "Hamiltonian"?
<details><summary><b>Show solution</b></summary>

Sketches:
```python
vl = {k: np.mean([voice_leading_distance(a, b) for s in v for a, b in zip(s, s[1:])]) for k, v in corpora.items()}
cover = {k: np.mean([len(set(s))/24 for s in v if s]) for k, v in corpora.items()}
```
The ranking persists: mean voice-leading cost is about 3.1 semitones for Bach and Beethoven and about 3.6 for the pop corpora, because a whole step between major triads (F→G) moves all three voices by 2 semitones (cost 6), while R-type moves cost 2. Coverage is well below 1 for all corpora: real pieces sit in small regions of the torus.

</details>